# Heart Disease — Inference Walkthrough

Loads the persisted `Pipeline` and demonstrates single + batch predictions, both directly via Python and against the FastAPI service.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.inference import load_model, predict_one, SAMPLE
model = load_model()
predict_one(model, SAMPLE)

In [ ]:
import pandas as pd
from src.preprocessing import FEATURE_COLUMNS
patients = pd.DataFrame([
    {'age':63,'sex':1,'cp':3,'trestbps':145,'chol':233,'fbs':1,'restecg':0,'thalach':150,'exang':0,'oldpeak':2.3,'slope':0,'ca':0,'thal':1},
    {'age':35,'sex':0,'cp':0,'trestbps':110,'chol':180,'fbs':0,'restecg':0,'thalach':190,'exang':0,'oldpeak':0.0,'slope':2,'ca':0,'thal':2},
    {'age':57,'sex':1,'cp':2,'trestbps':130,'chol':236,'fbs':0,'restecg':2,'thalach':174,'exang':0,'oldpeak':0.0,'slope':1,'ca':1,'thal':2},
])[FEATURE_COLUMNS]
probas = model.predict_proba(patients)[:, 1]
patients_out = patients.copy()
patients_out['probability'] = probas
patients_out['prediction'] = (probas >= 0.5).astype(int)
patients_out[['age','sex','cp','prediction','probability']]

## Hit the live API
Start the API in another terminal: `uvicorn src.api:app --port 8000`

In [ ]:
import requests
try:
    r = requests.post('http://127.0.0.1:8000/predict', json=SAMPLE, timeout=2)
    print(r.status_code, r.json())
except Exception as e:
    print('API not running:', e)